Testing scripting to transform csv into json for navigation
Secondarily to also transform to MARC files

In [1]:
import pandas as pd
import json
import math
import csv
from collections import defaultdict

In [ ]:
# read in as dict
navj = {}
cats = list()
subcats = list()

with open('data/eld-test.csv', newline='') as csvfile:
    dreader = csv.DictReader(csvfile)
    for row in dreader:
        cat = row['cat']
        subcat = row['subcat']
        name = row['GEOSERVER_NAME_AND_TITLE']
        if cat != '':
            if cat in cats:
                if subcat != '':
                    if subcat in subcats:
                        navj[cat][subcat][name] = {'pub':row['AUTHOR'],'title':row['VIEWER_TITLE'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}
                    else:
                        navj[cat][subcat] = {}
                        subcats.append(cat)
                        navj[cat][subcat][name] = {'pub':row['AUTHOR'],'title':row['VIEWER_TITLE'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}
                else:
                    navj[cat][name] = {'pub':row['AUTHOR'],'title':row['VIEWER_TITLE'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}
            else:
                navj[cat] = {}
                cats.append(cat)             
                if subcat != '':
                        navj[cat][subcat] = {}
                        subcats.append(cat)
                        navj[cat][subcat][name] = {'pub':row['AUTHOR'],'title':row['VIEWER_TITLE'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}
                else:
                    navj[cat][name] = {'pub':row['AUTHOR'],'title':row['VIEWER_TITLE'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}
print(navj)

{'Boundaries': {'NSAREA_SCOTLAND': {'pub': 'NatureScot', 'title': 'NatureScot Area Boundaries', 'date': '2023', 'area': 'Scotland', 'description': 'This dataset details the new NatureScot National Operations Areas, as of April 2023. These new areas replace the previous NatureScot SNH Areas (Post April 2011) boundaries. This dataset was digitised using the Ordnance Survey Boundary-Line open data, using the local authorities geometries.'}, 'Boundary-Line': {'pub': 'Ordnance Survey', 'title': 'Boundary-Line', 'date': '2026', 'area': 'England, Scotland, Wales', 'description': 'Ordnance Survey Boundary-Line is a free, quality-assured dataset covering the hierarchy of administrative and electoral boundaries across Great Britain, including current boundary names and a regularly updated national reference layer. Its main value is that it gives organisations a consistent, authoritative spatial framework for mapping and analysis: it helps people combine boundaries with census, demographic, envir

In [ ]:
with open("assets/nav2.json", "w") as f:
    json.dump(navj, f, indent=4)

In [6]:
print(navj)

{'Boundaries': {'NSAREA_SCOTLAND': {'pub': 'NatureScot', 'title': 'NatureScot Area Boundaries', 'date': '2023', 'area': 'Scotland', 'description': 'This dataset details the new NatureScot National Operations Areas, as of April 2023. These new areas replace the previous NatureScot SNH Areas (Post April 2011) boundaries. This dataset was digitised using the Ordnance Survey Boundary-Line open data, using the local authorities geometries.'}, 'Boundary-Line': {'pub': 'Ordnance Survey', 'title': 'Boundary-Line', 'date': '2026', 'area': 'England, Scotland, Wales', 'description': 'Ordnance Survey Boundary-Line is a free, quality-assured dataset covering the hierarchy of administrative and electoral boundaries across Great Britain, including current boundary names and a regularly updated national reference layer. Its main value is that it gives organisations a consistent, authoritative spatial framework for mapping and analysis: it helps people combine boundaries with census, demographic, envir

In [4]:
# pandas, read in data file and display top rows for verification 

data_df = pd.read_csv('data/eld-test.csv')
data_df.PUB_DATE = data_df.PUB_DATE.astype('Int64')
data_df.head(2)

,cat,subcat,AUTHOR,GEOSERVER_NAME_AND_TITLE,VIEWER_TITLE,PUB_DATE,AREA,MIN_X,MIN_Y,MAX_X,MAX_Y,NOTES,OpenData
0,Boundaries,NaN,NatureScot,NSAREA_SCOTLAND,NatureScot Area Boundaries,2023,Scotland,-9.230105,54.513188,-0.707475651,60.865310,This dataset details the new NatureScot Nation...,Y
1,Boundaries,NaN,Ordnance Survey,Boundary-Line,Boundary-Line,2026,"England, Scotland, Wales",-9.230654,49.818201,2.701860792,60.863633,"Ordnance Survey Boundary-Line is a free, quali...",Y


In [15]:
print(cats)

['Boundaries', 'Heritage sites', 'Natural environment']


In [16]:
print(subcats)

[]


In [ ]:
with open('data/eld-test.csv', newline='') as csvfile:
    dreader = csv.DictReader(csvfile)
    for row in dreader:
        cat = row['cat']
        subc = row['subcat']
        print("test:",cat,subc)

test: Boundaries 
test: Boundaries 
test: Boundaries 
test: Heritage sites Listed buildings
test: Heritage sites Listed buildings
test: Heritage sites Marine environment
test: Heritage sites Military sites
test: Heritage sites Monuments
test: Natural environment 
test: Natural environment 
test: Natural environment 


In [ ]:
# Create json file from csv

In [17]:
navlist = list()
cats = list()
subcats = list()

with open('data/eld-test.csv', newline='') as csvfile:
    dreader = csv.DictReader(csvfile)
    for row in dreader:
        cat = row['cat']
        subc = row['subcat']
        cid = row['GEOSERVER_NAME_AND_TITLE']
        name = row['VIEWER_TITLE']
        if cat != '':
            if cat in cats:
                # cat in list
                if subc != "":
                    # has subcategory
                    if subc in subcats:
                        for i in navlist:
                            if i["catname"] == cat: 
                                subs = i["subcats"]
                                for sc in subs:
                                    scname = sc["name"]
                                    if sc["name"] == subc:
                                        sc["maps"].append({'pub':row['AUTHOR'],'title':row['VIEWER_TITLE'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']})             
                    else:
                         # create subcat
                        if i["catname"] == cat: 
                            subs = i["subcats"]
                            subs.append([{"name": subc,'maps':[{'id':cid,'title':name,'pub':row['AUTHOR'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}] }] )  
                            subcats.append(subc)                        
                else:
                    # no subcats add to cat maps 
                        for i in navlist:
                            if i["catname"] == cat:
                                i["maps"].append({'id':cid,'title':name,'pub':row['AUTHOR'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']})
            else:           
                if subc != '':
                    navlist.append({'catname':cat,'id':cid,'subcats': [{"name": subc,'maps':[{'id':cid,'title':name,'pub':row['AUTHOR'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}] }] })
                    cats.append(cat)
                    subcats.append(subc)
                else:
                    navlist.append({'catname':cat,'id':cid,'maps':[{'id':cid,'title':name,'pub':row['AUTHOR'],'date':row['PUB_DATE'],'area':row['AREA'],'description':row['NOTES']}] })
                    cats.append(cat)  

In [16]:
with open("assets/navtest.json", "w") as f:
    json.dump(navlist, f, indent=4)

In [11]:
# select the rows which are open data, to convert to json
# nav = {}

# open_data =  data_df[data_df['OpenData'] == 'Y']
# catslist = open_data['cat'].unique() 
# print(catslist)
# for i in catslist: 
#    cat_data =  open_data[open_data['cat'] == i]
#    # print(cat_data)
#    subcats = cat_data['subcat'].unique()
#    print(len(subcats),subcats[0],i)
#    if (len(subcats) == 1) and math.isnan(subcats[0]):
#       sclist = getnavitems(cat_data)
#       nav[i] = sclist
#    else: 
#       sclist = []
#       for s in subcats:
#          scdf = cat_data[cat_data['subcat'] == s]
#          slist = getnavitems(scdf)
#          st = {s:sclist}
#          nav[i] = st
# print(nav)


# print(navlist)

# if needed, add nav thing
# site_json = {"navigation:":[catdict]}

